In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql.functions import col, count, when, isnan, countDistinct
from pyspark.sql import SparkSession

jar_paths = [
    "/home/jovyan/work/jars/delta-spark_2.12-3.1.0.jar",
    "/home/jovyan/work/jars/delta-storage-3.1.0.jar",
    "/home/jovyan/work/jars/hadoop-aws-3.3.4.jar",
    "/home/jovyan/work/jars/aws-java-sdk-bundle-1.12.262.jar"
]
jars_string = ",".join(jar_paths)

spark = SparkSession.builder \
    .appName("LakehouseSetup_Offline") \
    .config("spark.jars", jars_string) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print(f"Spark Version: {spark.version}")
print("Connected successfully in Offline Mode! Ready to build the Lakehouse.")

Spark Version: 3.5.0
Connected successfully in Offline Mode! Ready to build the Lakehouse.


Schema Enforcement

In [2]:
from pyspark.sql.utils import AnalysisException
from pyspark.sql.functions import lit

df_real_customers = spark.read.format("delta").load("s3a://olist-data/silver/customers").limit(5)
demo_path = "s3a://olist-data/demo/olist_schema_test"
df_real_customers.write.format("delta").mode("overwrite").save(demo_path)
print("-> Created Demo table (5 cols)")

# 2. Giả lập luồng dữ liệu mới tràn về từ Kafka/API nhưng bị THÊM 1 CỘT (loyalty_points)
print("\n Simulating newly received data, the Marketing team secretly added a 'loyalty_points' column...")
# Lấy thêm 1 dòng khác từ bảng thật và nhét thêm cột loyalty_points = 500
df_incoming_bad = spark.read.format("delta").load("s3a://olist-data/silver/customers") \
    .limit(1) \
    .withColumn("loyalty_points", lit(500))

# 3. Chứng minh Delta Lake sẽ CHẶN hành động này lại
print("-> Attempting to write (append) new data to the Demo table...")
try:
    df_incoming_bad.write.format("delta").mode("append").save(demo_path)
except AnalysisException as e:
    print("SCHEMA ENFORCEMENT BLOCKED! Structural mismatch:")
    print(f"-> {str(e).split(';')[0]}") 

# 4. Cách giải quyết (Schema Evolution)
df_incoming_bad.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .save(demo_path)

print("Writing successful! Delta Lake has automatically evolved the table to include the additional column 'loyalty_points'.")
print("(Previous customers without these points will automatically receive a NULL value)")

spark.read.format("delta").load(demo_path) \
    .select("customer_id", "customer_city", "loyalty_points").show()

-> Created Demo table (5 cols)

 Simulating newly received data, the Marketing team secretly added a 'loyalty_points' column...
-> Attempting to write (append) new data to the Demo table...
SCHEMA ENFORCEMENT BLOCKED! Structural mismatch:
-> A schema mismatch detected when writing to the Delta table (Table ID: 48229032-e335-48c6-8d72-f93208181ec1).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set:
'.option("mergeSchema", "true")'.
For other operations, set the session configuration
spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation
specific to the operation for details.

Table schema:
root
-- customer_id: string (nullable = true)
-- customer_unique_id: string (nullable = true)
-- customer_zip_code_prefix: integer (nullable = true)
-- customer_city: string (nullable = true)
-- customer_state: string (nullable = true)


Data schema:
root
-- customer_id: string (nullable = true)
-- customer_unique_id: string (nullable = true)
-

Time travel

In [3]:
df_history = spark.sql("DESCRIBE HISTORY delta.`s3a://olist-data/demo/olist_schema_test`")
df_history.select("version", "timestamp", "operation", "operationParameters").show(truncate=False)

# 2. Time Travel to Past Version
print("\nTIME TRAVEL: Querying Version 0 (Original schema)")
df_past = spark.read.format("delta").option("versionAsOf", 0).load("s3a://olist-data/demo/olist_schema_test")
df_past.show()

# View Current Version
print("\nCURRENT STATE: Querying Version 1 (After Schema Evolution)")
df_present = spark.read.format("delta").option("versionAsOf", 1).load("s3a://olist-data/demo/olist_schema_test")
df_present.show()

# 3. Perform Rollback
print("\nROLLBACK: Restoring data to Version 0")
df_past.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("s3a://olist-data/demo/olist_schema_test")

print("Rollback successful. A new version is automatically appended to the history.")
spark.read.format("delta").load("s3a://olist-data/demo/olist_schema_test").show()

+-------+-------------------+---------+--------------------------------------+
|version|timestamp          |operation|operationParameters                   |
+-------+-------------------+---------+--------------------------------------+
|1      |2026-05-07 14:49:20|WRITE    |{mode -> Append, partitionBy -> []}   |
|0      |2026-05-07 14:49:19|WRITE    |{mode -> Overwrite, partitionBy -> []}|
+-------+-------------------+---------+--------------------------------------+


TIME TRAVEL: Querying Version 0 (Original schema)
+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|           